# 01｜ブラウザからAPIへ：作品の書誌情報をまとめて調べる

## なぜAPIを使うのか？

Open Libraryのウェブページでは、作品や個々のEditionの
情報を読むことができます。

しかし、数十件・数百件のEditionについて、
刊行年・出版社・言語を一件ずつ確認してまとめるのは大変です。

そこでAPIを使います。APIから取得した書誌データを
Pythonで整理すると、複数のEditionをまとめて比較できます。

### この実習でやること

1. 作品を検索し、Open Libraryのページをブラウザで確認する。
2. 同じ作品のEditionデータをAPIから取得する。
3. APIで取得したEditionを年代・出版社・言語別にまとめる。
4. ダンプ由来の固定データと比較し、取得方法の違いを考える。

**データの違いに注意：**
APIから取得した100件と、ダンプ由来の全登録Edition集計は別です。
同じWorkでも、取得時点や収録範囲が異なる場合があります。

作品を変えた場合は、最初の検索から順番に再実行してください。


## ① 作品名・著者でWorkを検索する
表の「選択番号」は表示順（1、2、3…）です。`OL12345W`の数字とは無関係です。


In [ ]:
# @title ① 作品を検索する {display-mode: "form"}
import json
import pandas as pd
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from IPython.display import display

def ol_api(path, params=None):
    url = "https://openlibrary.org" + path
    if params:
        url += "?" + urlencode(params)
    req = Request(
        url,
        headers={"User-Agent": "LiteraryMetadataWorkshop2026/0.1"}
    )
    with urlopen(req, timeout=25) as response:
        return json.load(response)

SEARCH_TITLE = "Ulysses" # @param {type:"string"}
SEARCH_AUTHOR = "" # @param {type:"string"}

if not SEARCH_TITLE.strip() and not SEARCH_AUTHOR.strip():
    raise ValueError("作品名または著者名を入力してください。")

params = {
    "limit": 10,
    "fields": "key,title,author_name,first_publish_year,edition_count",
}
if SEARCH_TITLE.strip():
    params["title"] = SEARCH_TITLE.strip()
if SEARCH_AUTHOR.strip():
    params["author"] = SEARCH_AUTHOR.strip()

results = ol_api("/search.json", params)
ol_docs = results.get("docs", [])

ol_results = pd.DataFrame([
    {
        "Work ID": d.get("key", "").replace("/works/", ""),
        "作品名": d.get("title", ""),
        "著者": " / ".join(d.get("author_name") or []),
        "OL初刊年": d.get("first_publish_year"),
        "登録Edition数": d.get("edition_count"),
    }
    for d in ol_docs
])

print("Open Library全体の検索結果")
print("該当件数（API報告値）:",
      results.get("numFound", results.get("num_found")))
ol_results.insert(0, "選択番号", range(1, len(ol_results) + 1))
display(ol_results)

print("注意：OL初刊年は、歴史上の原作初刊年と一致するとは限りません。")

## ② Workを選んでEditionを取得する
「選択番号」をBの `SEARCH_RESULT_NUMBER` に入力、またはWork IDを直接指定します。


In [ ]:
# @title ② Workを選びEditionを取得する {display-mode: "form"}
import re

# 空欄なら直前の検索結果の先頭Workを選択
SELECT_WORK_ID = "" # @param {type:"string"}
SEARCH_RESULT_NUMBER = 1 # @param {type:"integer", min:1, max:10}
MAX_EDITIONS = 100 # @param {type:"integer", min:10, max:200}

if SELECT_WORK_ID.strip():
    wid = SELECT_WORK_ID.strip().replace("/works/", "")
elif not ol_results.empty:
    index = int(SEARCH_RESULT_NUMBER) - 1
    if not 0 <= index < len(ol_results):
        raise ValueError("検索結果の番号を確認してください。")
    wid = str(ol_results.iloc[index]["Work ID"])
else:
    raise ValueError("検索結果がありません。Work IDを指定してください。")

if not re.fullmatch(r"OL[0-9]+W", wid):
    raise ValueError("Work IDの形式を確認してください: " + wid)

limit = max(1, min(int(MAX_EDITIONS), 200))

response = ol_api(
    f"/works/{wid}/editions.json",
    {"limit": limit, "offset": 0},
)

def edition_language_codes(record):
    return [
        v["key"].rsplit("/", 1)[-1]
        for v in (record.get("languages") or [])
        if isinstance(v, dict) and isinstance(v.get("key"), str)
    ]

edition_rows = []
for item in response.get("entries", []):
    edition_rows.append({
        "Edition ID": item.get("key", "").replace("/books/", ""),
        "タイトル": item.get("title", ""),
        "刊行年・日付": item.get("publish_date", ""),
        "言語コード": ", ".join(edition_language_codes(item)),
        "出版社": ", ".join(item.get("publishers") or []),
    })

edition_view = pd.DataFrame(edition_rows)
total = response.get("size")
n = len(edition_rows)

print("Work ID:", wid)
print("APIが報告する登録Edition総数:", total)
print("今回取得したEdition数:", n)

if isinstance(total, int) and n < total:
    print("注意：一部のEditionのみ取得しています。"
          "以下の結果は全Editionの集計ではありません。")

display(edition_view.head(30))

if not edition_view.empty:
    print("取得したEditionに付いている言語タグの例:")
    display(
        edition_view["言語コード"]
        .replace("", "未記録")
        .value_counts()
        .head(15)
        .to_frame("取得範囲内のレコード数")
    )

print("この表は検索時点のAPIデータです。固定ダンプの集計とは異なります。")

## ③ APIで何が楽になるのか？

同じWorkをブラウザとAPIの両方で開いてみましょう。
ブラウザでは個々の記録を読み、APIからは複数のEditionを取得して、刊行年代・出版社・言語を一度に集計します。

**重要：この集計は今回APIで取得した範囲のみです。**


In [ ]:
# @title ③ ブラウザとAPIの取得結果を比較する {display-mode: "form"}
from collections import Counter
from IPython.display import Markdown, display
import pandas as pd
import re

work_url = f"https://openlibrary.org/works/{wid}"
api_url = (
    f"https://openlibrary.org/works/{wid}/editions.json"
    f"?limit={limit}&offset=0"
)

print("=== 同じWorkを2通りの方法で見る ===")
display(Markdown(
    f"**ブラウザの作品ページ：** [開く]({work_url})"
))
display(Markdown(
    f"**APIのJSONデータ：** [開く]({api_url})"
))

entries = response.get("entries", [])

print("\n=== APIでまとめて取得したEdition ===")
print("Work ID:", wid)
print("APIが報告するEdition総数:", response.get("size"))
print("今回取得したEdition数:", len(entries))

reported = response.get("size")
if isinstance(reported, int) and reported > 0:
    print(
        "今回取得した割合:",
        f"{len(entries) / reported:.1%}"
    )
    print(
        "注意：これはAPIの先頭から取得した記録であり、"
        "無作為標本ではありません。"
    )

decades = Counter()
publishers = Counter()
languages = Counter()
undated = 0

for record in entries:
    date_text = str(record.get("publish_date") or "")
    match = re.search(
        r"(?<!\d)(1[5-9]\d{2}|20\d{2})(?!\d)",
        date_text
    )
    if match:
        year = int(match.group(1))
        decades[(year // 10) * 10] += 1
    else:
        undated += 1

    for name in set(record.get("publishers") or []):
        if isinstance(name, str) and name.strip():
            publishers[name.strip()] += 1

    codes = set(edition_language_codes(record))
    for language in (codes or {"言語タグなし"}):
        languages[language] += 1

def show_counts(counter, key, maximum=12):
    rows = [
        {key: k, "Edition登録数": v}
        for k, v in counter.most_common(maximum)
    ]
    display(pd.DataFrame(rows, columns=[key, "Edition登録数"]))

print("\n① 刊行年代別（取得範囲内）")
display(pd.DataFrame([
    {
        "年代": f"{decade}s",
        "Edition登録数": count,
    }
    for decade, count in sorted(decades.items())
]))
print("刊行年を判読できなかったEdition:", undated)

print("\n② 出版社別（上位12件）")
show_counts(publishers, "出版社")

print("\n③ 登録言語別（上位12件）")
show_counts(languages, "言語コード")

total = response.get("size")
if not isinstance(total, int) or len(entries) < total:
    print(
        "\n【注意】今回はAPIの一部のEditionだけを取得しています。"
        "この集計は全Editionの分布ではありません。"
    )
else:
    print(
        "\nAPIが報告した件数と取得件数が一致しています。"
    )

print(
    "出版社や言語が複数登録されたEditionは、"
    "それぞれの分類に計上します。"
)
print(
    "ブラウザは個別記録の確認に向き、"
    "APIは複数記録の取得と整理に向いています。"
)


## ④ ダンプ由来の集計と比較する

ここまでのAPI取得結果とは別に、
2026年2月28日版のOpen Libraryダンプから作成した
言語・刊行年別の集計を読み込みます。

固定集合34,789件に含まれるWorkでは、
APIの取得上限に左右されない固定スナップショットの集計を利用します。
含まれないWorkでは、先ほどAPIから取得した範囲だけを使います。

言語の違いは、書誌データをまとめて取得することで
調べられることの一例です。

ただしEdition数は読者数や売上ではなく、
Workの分裂・誤紐づけ・言語タグの欠損にも注意が必要です。


In [ ]:
# @title ④ 出版年代と言語を分析する {display-mode: "form"}
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# このCSVは後でGitHubに公開します。まずは候補Notebookとして構築します。
LANGUAGE_YEAR_URL = (
    "https://raw.githubusercontent.com/"
    "haruka0221/literary-metadata-workshop-2026/"
    "main/data/work_edition_language_year_v1.csv"
)
lang_history = pd.read_csv(LANGUAGE_YEAR_URL, dtype=str, keep_default_na=False, encoding="utf-8-sig")
assert {"ol_work_id", "language_code", "edition_publish_year", "edition_count"} <= set(lang_history.columns)

fixed = lang_history[lang_history["ol_work_id"].eq(wid)].copy()
if not fixed.empty:
    source_label = "2026-02-28の固定ダンプ：該当Workの登録レコード"
    raw = fixed.rename(columns={"edition_publish_year":"year_string", "edition_count":"n"})[
        ["language_code", "year_string", "n"]
    ].copy()
    raw["language_code"] = raw["language_code"].replace("", "no_tag")
    raw["n"] = pd.to_numeric(raw["n"], errors="raise")
else:
    source_label = "ライブAPI：取得できたEditionのみ（部分集計の場合あり）"
    api_rows = []
    for _, record in edition_view.iterrows():
        found = re.search(r"(?<!\d)(1[5-9]\d{2}|20\d{2})(?!\d)", str(record["刊行年・日付"]))
        year = found.group(1) if found else "UNKNOWN"
        languages = list(dict.fromkeys(x.strip() for x in str(record["言語コード"]).split(",") if x.strip()))
        for lang in (languages or ["no_tag"]):
            api_rows.append({"language_code":lang,"year_string":year,"n":1})
    raw = pd.DataFrame(api_rows, columns=["language_code", "year_string", "n"])

raw["year"] = pd.to_numeric(raw["year_string"].replace("UNKNOWN", None), errors="coerce")
known_names = {
    "eng":"英語", "jpn":"日本語", "fre":"フランス語", "fra":"フランス語",
    "ger":"ドイツ語", "deu":"ドイツ語", "spa":"スペイン語", "ita":"イタリア語",
    "rus":"ロシア語", "chi":"中国語", "zho":"中国語", "por":"ポルトガル語",
    "dut":"オランダ語", "nld":"オランダ語", "pol":"ポーランド語",
    "swe":"スウェーデン語", "dan":"デンマーク語", "nor":"ノルウェー語",
    "ara":"アラビア語", "hin":"ヒンディー語", "kor":"韓国語", "lat":"ラテン語",
    "no_tag":"言語未記録",
}
available = (raw.groupby("language_code")["n"].sum().sort_values(ascending=False))
print("選択したWork:", wid)
print("集計の出典:", source_label)
if fixed.empty:
    print("API報告総数:", response.get("size"), "/ 今回取得:", len(edition_view))
print("このWorkで発見された言語コード（入力に利用できます）:")
summary_lang = available.rename_axis("言語コード").reset_index(name="登録数（言語別）")
summary_lang["言語名（主要言語のみ）"] = summary_lang["言語コード"].map(known_names).fillna("コード参照")
display(summary_lang)

# 空欄なら、最も多く登録された言語を自動選択。例: eng,jpn または all
LANGUAGE_CODES = "" # @param {type:"string"}
PERIOD = "全期間（2026年まで）" # @param ["全期間（2026年まで）", "1880–1950", "1951–2026"]
ranges = {"全期間（2026年まで）":(1500,2026), "1880–1950":(1880,1950), "1951–2026":(1951,2026)}
start, end = ranges[PERIOD]
selected = [s.strip().lower() for s in LANGUAGE_CODES.split(",") if s.strip()]
if not selected:
    selected = available.index[:1].tolist()
elif selected == ["all"]:
    selected = available.index[:8].tolist()
    print("all：登録数上位8言語を表示します。全言語は上の表で確認できます。")
selected = list(dict.fromkeys(selected))
invalid = [c for c in selected if c not in available.index]
if invalid:
    print("このWorkで見つからなかった言語コード:", invalid)
    selected = [c for c in selected if c in available.index]
if selected:
    portion = raw[raw["language_code"].isin(selected) & raw["year"].between(start,end)].copy()
    portion["decade"] = (portion["year"] // 10 * 10).astype(int)
    chart = portion.pivot_table(index="decade", columns="language_code", values="n", aggfunc="sum", fill_value=0)
    if not portion.empty:
        decade_index = list(range(int(portion["decade"].min()), int(portion["decade"].max())+1, 10))
        chart = chart.reindex(index=decade_index,columns=selected,fill_value=0)
        display(chart.T.rename_axis("言語コード"))
        ax=chart.plot(figsize=(11,4.5), marker="o")
        ax.set(xlabel="Recorded publication decade", ylabel="Edition records",
               title="Publication record timeline (by recorded language)")
        ax.grid(axis="y",alpha=0.2)
        plt.tight_layout(); plt.show()
    else:
        print("指定した言語・期間に刊行年判読可能な記録がありません。")
    print("選択期間の刊行年判読可能件数（言語別）:", int(portion["n"].sum()))
    print("この集計で刊行年不明の件数（選択言語）:", int(raw.loc[raw["language_code"].isin(selected) & raw["year"].isna(), "n"].sum()))
else:
    print("表示する言語がありません。上の表に記載されたコードを入力してください。")
print("注意：複数言語タグを持つEditionは言語ごとに計上され、言語別件数の合計では重複します。")
print("2026年はデータ取得時点までの記録であり、未来日付・誤登録があり得ます。")


## ④ 何が言えて、何が言えないか
- 意外な言語のEditionは見つかりましたか。いつ現れていますか。
- API取得件数はWorkの総件数と一致していますか。
- 「Editionが多い」ことは読者の多さや翻訳数を示しますか。
- Work分裂・誤紐づけ・言語タグ未記録の影響をどう検証しますか。
